#NewsLens
Structured News Understanding via LLM Distillation and LoRA

In [ ]:
from google.colab import drive
drive.mount('/gdrive')


Mounted at /gdrive


# setup Libraries

In [ ]:
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1
!pip install -qU faker==35.2.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import wandb

wandb.login(key = '....')
hf_token = '......'
!huggingface-cli login --token {hf_token}

In [ ]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime


from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && pip install -e .  && pip install -r requirements/metrics.txt

Dawnload Dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("abisee/cnn_dailymail", "3.0.0")

In [ ]:
data  = ds['train']['article'][:2500]
print (len(data))

2500


# DETAILS EXTRACTION


In [ ]:
StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):

  title : str = Field (... , min_length=3 , max_length=100 , description= 'A fully informative optimized title of the story.')

  story_keywords: List[str] = Field(..., min_items=1,
                                      description="Relevant keywords associated with the story.")

  story_summary: List[str] = Field(..., min_items=1, max_items=5,                           # summrize the keypoints of the story in 5 points
                                    description="Summarized key points about the story (1-5 points).")

  story_category : StoryCategory= Field(..., description = "Category of the new story" )


  story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")


In [ ]:

def details_Extract_messages(article):

    return [
      {
      "role": "system",
      "content": "\n".join(["You are a professional information extractor.",
                "You will be provided by an English text associated with a Pydantic scheme.",
                "Generate the ouptut in the same story language.",
                "You have to extract JSON details from text according the Pydantic details.",
                "Extract details as mentioned in text.",
                "Do not generate any introduction or conclusion."
            ])

      },
      {
          "role": "user" ,
          "content": "\n".join([
            # Article :
            article.strip() ,
            "",
            # Pydatntic Details :
            json.dumps(NewsDetails.model_json_schema() , ensure_ascii = False) ,
            "",
            #....Story Details :
            "```json"

          ])
      }
    ]

# Knowledge Distillation  


**Get the model & Tokenizer from Hugging_face**

In [ ]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map = "auto",
    torch_dtype = None )  # for quantization

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)


**build_prompt** : Convert message sample to exact format that the model excpect

**Process_batch** :
- 1/ tokenize samples
- 2/ Pass it to the model
- 3/ Extract the outputs_ids from the model ouput
- 4/ decode the model outputs_ids  back to Json formate  

In [ ]:
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# apply chat template function

def build_prompt(article):

  message = details_Extract_messages(article)

  return tokenizer.apply_chat_template(
          message ,
          tokenize = False ,   # keep the template
          add_generation_prompt=True  # Let the model generate output
      )

# apply build prompt function for all data
all_inputs = [build_prompt(ex) for ex in data]


def process_batch(batch_text):

  # tokenize batch
  model_inputs = tokenizer(
      batch_text ,
      padding = True ,
      truncation = True ,
      return_tensors = 'pt').to(model.device)

  # pass tokenized batch to the model

  generate_ids = model.generate(
          **model_inputs,              # pass the input_ids & mask
          max_new_tokens=512,           # generate at max 1024 token
        do_sample=False, top_k=None, temperature=None, top_p=None,  #  control the words selection (greedy , beam ..)
  ).to(model.device)

  # split the outputs_ids only   ( the model mix the inputs and outputs ids togheter )

  generate_ids = [outputs_ids[len(inputs_ids):]
      for inputs_ids , outputs_ids in zip (model_inputs.input_ids , generate_ids)
  ]
  # decode output to normal text
  response = tokenizer.batch_decode(generate_ids , skip_special_tokens = True)

  return response

In [ ]:
import json
import os
from tqdm.auto import tqdm

batch_size = 4
save_path = '/gdrive/MyDrive/NewsLens/distilled_dataset.json'
failed_path = '/gdrive/MyDrive/NewsLens/Faild_dataset.json'

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(save_path), exist_ok=True)
os.makedirs(os.path.dirname(failed_path), exist_ok=True)

results = []
failed = []

for i in tqdm(range(590 , len(all_inputs) , batch_size)):

  batch_prompt = all_inputs[i : i + batch_size]
  batch_articles = data[i : i + batch_size]

  responses = process_batch(batch_prompt)

  # parse and save each batch
  for article, response in zip(batch_articles, responses):
        try:
            raw = response.strip().removesuffix("```").strip()
            result = NewsDetails.model_validate_json(raw)  # validate the respnse on The pydantic Schema
            entry = result.model_dump()                    # convert it to plain dict
            entry["original_text"] = article               # add the original article to the dict
            results.append(entry)

        except Exception as e:
            failed.append({
                "original_text": article,
                "error": str(e)                            # ADD error att to the sample dict so we can cooun it later
            })

  if (i // batch_size) % 25 == 0 :
    with open(save_path , 'w') as f :
      json.dump(results , f,ensure_ascii = False , indent = 2)

    with open(failed_path , 'a') as h :
      json.dump(failed , f,ensure_ascii = False , indent = 2)

with open(save_path , 'w') as f :
  json.dump(results , f , ensure_ascii = False , indent =2)

print (f"!done {len(results)} saved examples ")
print (f"Faild {len(failed)}")

  0%|          | 0/478 [00:00<?, ?it/s]

# Format Fine-tuning Dataset

Convert the dataset to format that llama_factory excpected

In [ ]:
with open ('/gdrive/MyDrive/NewsLens/distilled_dataset.json' , 'r' , encoding='utf8') as f :
  data = json.load(f)

llama_dataset = []
system_message =  "\n".join(["You are a professional information extractor.",
                "Extract details as mentioned in text.",
                "Do not generate any introduction or conclusion."])

for example in data :
  llama_dataset.append({
      'system':system_message ,
      'instruction' :"Extract structured information from the given news article as a JSON object.",
      'input': example['original_text'],
      'output': json.dumps({
            "title": example['title'],
            "story_category": example["story_category"],
            "story_summary": example["story_summary"],
            "story_entities": example["story_entities"],
            "story_keywords": example["story_keywords"]
      }, ensure_ascii=False)

      })

llama_savepath = '/gdrive/MyDrive/NewsLens/model/Llama_dataset.json'
os.makedirs(os.path.dirname(llama_savepath) , exist_ok=True)

# shuffle the data
random.Random(101).shuffle(llama_dataset)

# save it
with open (llama_savepath , 'w' , encoding='utf8')as f:
  json.dump(llama_dataset ,f ,ensure_ascii=False, indent=2)

print (f" {len(llama_dataset)}llama examples saved ")

# Fine-tuning  

In [ ]:
# # Configure LLaMA-Factory for the new datasets
# first we open LamaFactory/data/dataset_info.json  and add this

  # "newslens": {
  #   "file_name": "/gdrive/MyDrive/NewsLens/models/Llama_dataset.json",
  #   "columns": {
  #     "prompt": "instruction",
  #     "query": "input",
  #     "response": "output"
  #   }
  # }


In [ ]:
%%writefile /content/LlamaFactory/examples/train_lora/Newslens.yaml
# criate and type on the file


### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft                                # supervised fine-tuning (input -> output) train
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all                          #use entire model

### dataset
dataset: newslens                         # dataset that ihave putted on data_info.json
# eval_dataset: news_finetune_val
template: qwen                            # from llama_factory repo  templates
cutoff_len: 3500                          # max tokens per example
max_samples: 10                           # run expermients
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/NewsLens/models/...
output_dir: /gdrive/MyDrive/NewsLens/models
logging_steps: 10                     # logging after 10 samples
save_steps: 500                       # checkpoint every 500 sample
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4           # update gradient every 4 batches
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
val_size: 0.1           # comment it if you have val_dataser
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb                                  # whight & bias
run_name: newslens-finetune-llamafactory

# uplode to  hugging face
# push_to_hub: true
# export_hub_model_id: "..."
# hub_private_repo: true
# hub_strategy: checkpoint



Overwriting /content/LlamaFactory/examples/train_lora/Newslens.yaml


**Fine-tune**

In [ ]:
# Run
# we pass to it the train folder that we criated above

!cd LlamaFactory/ && llamafactory-cli train /content/LlamaFactory/examples/train_lora/Newslens.yaml

# Inference

- load same model (qwen1.5B) & TOkenizer again
- Load the saved adapter from Hugging_Face or gdrive
- use Model.load_adapter = 'saved_adapter name '  


In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)



In [ ]:
fine_tuned_model = '/gdrive/MyDrive/NewsLens/models'

# Connect The  adapter to the model
model.load_adapter(fine_tuned_model)

In [ ]:
def inference(article):
  message =  details_Extract_messages(article)

  text = tokenizer.apply_chat_template(
      message ,
      tokenize = False ,
      add_generation_prompt=True

  )

  model_inputs = tokenizer([text], return_tensors="pt").to(device)

  generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

  generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

  response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

  response.strip().removesuffix("```").strip()

  return response

response = inference(data[1])
